In [1]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"D:\RouteRanker")
DELHI_DIR = PROJECT_DIR / "data" / "raw" / "delhi"

print("Project folder exists:", PROJECT_DIR.exists())
print("Delhi folder exists:", DELHI_DIR.exists())

# Find GTFS files, even if the downloaded folder is nested
for file in DELHI_DIR.rglob("*.txt"):
    print(file.relative_to(DELHI_DIR))


Project folder exists: True
Delhi folder exists: True
bus\agency.txt
bus\calendar.txt
bus\fare_attributes.txt
bus\fare_rules.txt
bus\routes.txt
bus\stops.txt
bus\stop_times.txt
bus\trips.txt


In [2]:

# Find the folder containing the GTFS files
gtfs_dirs = [
    folder
    for folder in DELHI_DIR.rglob("*")
    if folder.is_dir()
    and (folder / "routes.txt").exists()
    and (folder / "stops.txt").exists()
    and (folder / "trips.txt").exists()
]

# Also check whether the files are directly inside DELHI_DIR
if all((DELHI_DIR / f).exists()
       for f in ["routes.txt", "stops.txt", "trips.txt"]):
    GTFS_DIR = DELHI_DIR
elif gtfs_dirs:
    GTFS_DIR = gtfs_dirs[0]
else:
    raise FileNotFoundError(
        "Could not find routes.txt, stops.txt and trips.txt together."
    )

print("GTFS folder:", GTFS_DIR)

routes = pd.read_csv(GTFS_DIR / "routes.txt", dtype=str)
stops = pd.read_csv(GTFS_DIR / "stops.txt", dtype=str)
trips = pd.read_csv(GTFS_DIR / "trips.txt", dtype=str)

print("\nRoutes shape:", routes.shape)
print("Stops shape:", stops.shape)
print("Trips shape:", trips.shape)


GTFS folder: D:\RouteRanker\data\raw\delhi\bus

Routes shape: (2403, 5)
Stops shape: (10559, 6)
Trips shape: (89393, 4)


In [3]:

print("ROUTES")
display(routes.head())

print("STOPS")
display(stops.head())

print("TRIPS")
display(trips.head())


ROUTES


,agency_id,route_id,route_long_name,route_short_name,route_type
0,DIMTS,142,828AUP,NaN,3
1,DIMTS,10001,971DOWN,NaN,3
2,DIMTS,362,961DOWN,NaN,3
3,DIMTS,274,819DOWN,NaN,3
4,DIMTS,10386,824STLDOWN2,NaN,3


STOPS


,stop_code,stop_id,stop_lat,stop_lon,stop_name,zone_id
0,DC4539,0,28.851958,77.088107,Narela Terminal,0
1,DC4540,1,28.853311,77.088548,Police Station Narela,1
2,DC4541,2,28.854965,77.089322,Safiyabad Crossing,2
3,DC4542,3,28.857713,77.092846,Ramdev Chowk / Pitori Johad,3
4,DC4543,4,28.857684,77.097117,Narela A-6 / CPJ College,4


TRIPS


,route_id,service_id,trip_id,shape_id
0,142,1,142_12_50,NaN
1,142,1,142_11_17,NaN
2,142,1,142_09_14,NaN
3,142,1,142_14_00,NaN
4,142,1,142_10_44,NaN


In [4]:

for name, df in [
    ("routes", routes),
    ("stops", stops),
    ("trips", trips)
]:
    print(f"\n--- {name.upper()} ---")
    print("Rows:", len(df))
    print("Duplicate rows:", df.duplicated().sum())
    print("\nMissing values per column:")
    print(df.isna().sum())



--- ROUTES ---
Rows: 2403
Duplicate rows: 0

Missing values per column:
agency_id              0
route_id               0
route_long_name        0
route_short_name    2403
route_type             0
dtype: int64

--- STOPS ---
Rows: 10559
Duplicate rows: 0

Missing values per column:
stop_code    0
stop_id      0
stop_lat     0
stop_lon     0
stop_name    0
zone_id      0
dtype: int64

--- TRIPS ---
Rows: 89393
Duplicate rows: 0

Missing values per column:
route_id          0
service_id        0
trip_id           0
shape_id      89393
dtype: int64


In [5]:

print("GTFS files found:")
for file in sorted(GTFS_DIR.glob("*.txt")):
    print(f"{file.name}: {file.stat().st_size:,} bytes")

if {"stop_lat", "stop_lon"}.issubset(stops.columns):
    stops["stop_lat"] = pd.to_numeric(
        stops["stop_lat"], errors="coerce"
    )
    stops["stop_lon"] = pd.to_numeric(
        stops["stop_lon"], errors="coerce"
    )

    print("\nStops with valid coordinates:",
          stops[["stop_lat", "stop_lon"]].notna().all(axis=1).sum())

    print("\nCoordinate summary:")
    display(stops[["stop_lat", "stop_lon"]].describe())
else:
    print("\nStop latitude/longitude columns were not found.")


GTFS files found:
agency.txt: 260 bytes
calendar.txt: 122 bytes
fare_attributes.txt: 133,029,360 bytes
fare_rules.txt: 84,146,786 bytes
routes.txt: 52,002 bytes
stop_times.txt: 142,098,714 bytes
stops.txt: 612,802 bytes
trips.txt: 1,788,264 bytes

Stops with valid coordinates: 10559

Coordinate summary:


,stop_lat,stop_lon
count,10559.000000,10559.000000
mean,28.641227,77.153683
std,0.080761,0.100400
min,28.416651,76.842772
25%,28.580853,77.082160
50%,28.636379,77.163492
75%,28.691698,77.227865
max,28.871866,77.414551


In [6]:

stop_times_path = GTFS_DIR / "stop_times.txt"

if stop_times_path.exists():
    stop_times = pd.read_csv(stop_times_path, dtype=str)

    print("Stop times shape:", stop_times.shape)
    print("Columns:", stop_times.columns.tolist())
    display(stop_times.head())
else:
    print("stop_times.txt was not found in", GTFS_DIR)


Stop times shape: (3724320, 5)
Columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence']


,trip_id,arrival_time,departure_time,stop_id,stop_sequence
0,142_12_50,12:50:00,12:50:00,146,0
1,142_12_50,12:51:11,12:51:11,148,1
2,142_12_50,12:52:49,12:52:49,149,2
3,142_12_50,12:53:54,12:53:54,488,3
4,142_12_50,12:55:44,12:55:44,233,4


In [7]:

# Check whether trips refer to valid routes
if {"route_id", "trip_id"}.issubset(trips.columns) and \
   {"route_id"}.issubset(routes.columns):

    print(
        "Trips referencing missing routes:",
        (~trips["route_id"].isin(routes["route_id"])).sum()
    )

# Check whether stop times reference valid trips
if stop_times_path.exists() and \
   {"trip_id"}.issubset(stop_times.columns):

    print(
        "Stop times referencing missing trips:",
        (~stop_times["trip_id"].isin(trips["trip_id"])).sum()
    )

# Check whether stop times reference valid stops
if stop_times_path.exists() and \
   {"stop_id"}.issubset(stop_times.columns):

    print(
        "Stop times referencing missing stops:",
        (~stop_times["stop_id"].isin(stops["stop_id"])).sum()
    )


Trips referencing missing routes: 0
Stop times referencing missing trips: 0
Stop times referencing missing stops: 0


In [8]:
print("===== DELHI GTFS DATASET SUMMARY =====")

print("Routes:", len(routes))
print("Stops:", len(stops))
print("Trips:", len(trips))

if "stop_times" in globals():
    print("Stop-time records:", len(stop_times))

print("\n===== REQUIRED FILES =====")
for filename in [
    "routes.txt",
    "stops.txt",
    "trips.txt",
    "stop_times.txt"
]:
    file_path = GTFS_DIR / filename
    print(f"{filename}: {'Available' if file_path.exists() else 'Missing'}")

===== DELHI GTFS DATASET SUMMARY =====
Routes: 2403
Stops: 10559
Trips: 89393
Stop-time records: 3724320

===== REQUIRED FILES =====
routes.txt: Available
stops.txt: Available
trips.txt: Available
stop_times.txt: Available


In [9]:

# Step 8: Check relationships between GTFS tables

print("===== GTFS RELATIONSHIP CHECKS =====")

# Trips should reference valid routes
print(
    "Trips with missing routes:",
    (~trips["route_id"].isin(routes["route_id"])).sum()
)

# Stop times should reference valid trips
print(
    "Stop times with missing trips:",
    (~stop_times["trip_id"].isin(trips["trip_id"])).sum()
)

# Stop times should reference valid stops
print(
    "Stop times with missing stops:",
    (~stop_times["stop_id"].isin(stops["stop_id"])).sum()
)

# Check whether each trip has stop-time records
trips_without_stop_times = (
    ~trips["trip_id"].isin(stop_times["trip_id"])
).sum()

print("Trips without stop-time records:", trips_without_stop_times)


===== GTFS RELATIONSHIP CHECKS =====
Trips with missing routes: 0
Stop times with missing trips: 0
Stop times with missing stops: 0
Trips without stop-time records: 0


In [10]:

print("===== STOP-TIME DATA QUALITY =====")

print("Missing values:")
display(stop_times.isnull().sum())

print(
    "Duplicate stop-time rows:",
    stop_times.duplicated().sum()
)

print("\nFirst five stop-time records:")
display(stop_times.head())


===== STOP-TIME DATA QUALITY =====
Missing values:


trip_id           0
arrival_time      0
departure_time    0
stop_id           0
stop_sequence     0
dtype: int64

Duplicate stop-time rows: 0

First five stop-time records:


,trip_id,arrival_time,departure_time,stop_id,stop_sequence
0,142_12_50,12:50:00,12:50:00,146,0
1,142_12_50,12:51:11,12:51:11,148,1
2,142_12_50,12:52:49,12:52:49,149,2
3,142_12_50,12:53:54,12:53:54,488,3
4,142_12_50,12:55:44,12:55:44,233,4
